# Journal revision visualizations

Unified analyses and figures for the journal revision.


In [ ]:
# ruff: noqa: I001,D103,B007,B905,E402,E741,W505
import gc
import gzip
import math
import pickle
import sys
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy.stats import mannwhitneyu

for candidate in [
	Path.cwd(),
	Path.cwd() / "revision",
	Path.cwd() / "examples" / "journal" / "revision",
]:
	if (candidate / "constant.py").exists():
		sys.path.insert(0, str(candidate.resolve()))
		break
else:
	raise FileNotFoundError("Could not locate examples/journal/revision/constant.py")

from constant import (
	BLACK,
	CHUNK_ROWS,
	DISTANCES,
	ELMD_AMD_DEFAULT_WEIGHT,
	ELMD_AMD_WEIGHT_MEAN_SCORES_PATH,
	ELMD_AMD_WEIGHT_METRICS,
	ELMD_AMD_WEIGHTS,
	FIGURES_DIR,
	FIG_WIDTH,
	FONT_L,
	FONT_M,
	FONT_S,
	GRAY,
	JOURNAL_DIR,
	MEAN_SCORES_PATH as NORMALIZED_MEAN_SCORES_PATH,
	METRIC_LABEL_DICTS,
	MODEL_TEST_LABELS,
	MODELS,
	MODELS_TEST,
	NORMALIZATION_KEYS,
	NORMALIZATION_LABELS,
	NORMALIZATION_PLOT_SETTINGS,
	NORMALIZATIONS,
	OVERWRITE,
	PALETTE,
	PAD_L,
	PAD_M,
	PAD_S,
	PREPROCESS_DIR,
	RESULTS_DIR,
	SPEARMAN_PATH as NORMALIZATION_SPEARMAN_PATH,
	STABILITY_INTERCEPT,
	WHITE,
)

JOURNAL_PREPROCESS_DIR = JOURNAL_DIR / "preprocess"

plt.rcParams["font.family"] = "cmr10"
plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams["axes.formatter.use_mathtext"] = True
sns.set_palette(PALETTE)


def load_gz(path):
	with gzip.open(path, "rb") as f:
		return pickle.load(f)


def score_mean(scores):
	values = np.asarray(scores, dtype=float).copy()
	values[np.isnan(values)] = 0.0
	return np.mean(values).item()


def continuous_stability_scores(ehulls, tau=STABILITY_INTERCEPT, exponent=1.0):
	linear = np.clip(1 - np.asarray(ehulls, dtype=float) / tau, 0, 1)
	return linear**exponent


def metric_label(metric, distance):
	return METRIC_LABEL_DICTS[metric][distance]


def plot_stripplot_panel(
	ax,
	df_means,
	filters,
	title,
	show_title=False,
):
	mask = np.ones(len(df_means), dtype=bool)
	for column, value in filters.items():
		mask &= df_means[column] == value
	df_sub = df_means[mask]
	sns.stripplot(
		data=df_sub[df_sub["Model"] != "test"],
		y="Mean score",
		hue="Model",
		hue_order=MODELS,
		dodge=True,
		size=4,
		edgecolor=BLACK,
		linewidth=1,
		ax=ax,
	)
	test_score = df_sub[df_sub["Model"] == "test"]["Mean score"].values[0]
	ax.axhline(y=test_score, color=GRAY, linestyle="--", label="test", dashes=(2, 1))
	y_data = df_sub["Mean score"].values
	max_val = y_data.max()
	min_val = y_data.min()
	margin = max(abs(max_val - test_score), abs(min_val - test_score), 0.005)
	ax.set_ylim(test_score - 1.08 * margin, test_score + 1.08 * margin)
	if test_score + 1.08 * margin > 1.0:
		ax.axhspan(
			1.0,
			test_score + 1.08 * margin,
			facecolor=WHITE,
			edgecolor=GRAY,
			hatch="//////",
			alpha=1.0,
			zorder=0,
		)
	if show_title:
		ax.set_title(title, fontsize=FONT_M, pad=PAD_M)
	total_span = margin * 2.1
	chosen_step = 0.01
	for step in [0.25, 0.2, 0.1, 0.05, 0.02, 0.01]:
		if (total_span / step) >= 3:
			chosen_step = step
			break
	ax.yaxis.set_major_locator(ticker.MultipleLocator(chosen_step))
	ax.set_yticks([t for t in ax.get_yticks() if 0 <= t <= 1 + 1e-3])
	ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
	ax.set_xticks([])
	ax.tick_params(axis="y", labelsize=FONT_S)
	ax.set_ylim(test_score - 1.08 * margin, test_score + 1.08 * margin)
	legend = ax.get_legend()
	if legend is not None:
		legend.remove()
	ax.set_xlabel("")
	ax.set_ylabel("")


def plot_sensitivity_stripplots(
	df_means,
	rows,
	columns,
	column_filters,
	column_titles,
	filename,
	figure_height,
	wspace,
	legend_y,
	labelpad,
):
	fig, axes = plt.subplots(
		len(rows),
		len(columns),
		figsize=(FIG_WIDTH, figure_height * FIG_WIDTH),
		gridspec_kw={"wspace": wspace, "hspace": 0.2},
	)
	legend_handles, legend_labels = None, None
	for row, (metric, distance) in enumerate(rows):
		for col, column in enumerate(columns):
			ax = axes[row, col]
			plot_stripplot_panel(
				ax,
				df_means,
				{
					"Metric": metric,
					"Distance": distance,
					**column_filters(column),
				},
				column_titles(column),
				show_title=(row == 0),
			)
			if row == 0 and col == 0:
				legend_handles, legend_labels = ax.get_legend_handles_labels()
			if col == 0:
				ax.set_ylabel(
					metric_label(metric, distance),
					fontsize=FONT_L,
					rotation=0,
					labelpad=labelpad,
				)
	legend_labels = [MODEL_TEST_LABELS[label] for label in legend_labels]
	n = len(legend_handles) // 2
	legend_handles = [
		h for pair in zip(legend_handles[:n], legend_handles[n:]) for h in pair
	]
	legend_labels = [
		l for pair in zip(legend_labels[:n], legend_labels[n:]) for l in pair
	]
	fig.legend(
		legend_handles,
		legend_labels,
		loc="lower center",
		bbox_to_anchor=(0.5, legend_y),
		ncol=len(MODELS_TEST) // 2,
		fontsize=FONT_L,
		handletextpad=0.3,
	)
	plt.savefig(FIGURES_DIR / filename, bbox_inches="tight")
	return fig


def plot_spearman_heatmaps(
	matrices,
	titles,
	filename,
	nrows,
	ncols,
	figure_height,
	wspace,
	hspace=None,
):
	gridspec_kw = {"wspace": wspace}
	if hspace is not None:
		gridspec_kw["hspace"] = hspace
	fig, axes = plt.subplots(
		nrows,
		ncols,
		figsize=(FIG_WIDTH, figure_height * FIG_WIDTH),
		gridspec_kw=gridspec_kw,
	)
	cbar_ax = fig.add_axes([0.92, 0.25, 0.015, 0.5])
	for i, (matrix, title) in enumerate(zip(matrices, titles, strict=True)):
		ax = np.asarray(axes).flat[i]
		sns.heatmap(
			matrix,
			vmin=-1,
			vmax=1,
			center=0,
			cmap="vlag",
			annot=True,
			fmt=".2f",
			square=True,
			cbar=(i == 0),
			cbar_ax=cbar_ax if i == 0 else None,
			ax=ax,
		)
		ax.set_title(title, fontsize=FONT_L, pad=PAD_M)
		ax.set_xlabel("")
		ax.set_ylabel("")
		ax.tick_params(axis="both", labelsize=FONT_S)
	plt.savefig(FIGURES_DIR / filename, bbox_inches="tight")
	return fig

## 1. SUN_smat and cSUN_elm+am probability of superiority

Compute the probability that a randomly selected MatterGen sample with
$\mathrm{SUN}_\mathrm{smat}=1$ has a larger
$\mathrm{cSUN}_\mathrm{elm{+}am}$ value than a randomly selected sample with
$\mathrm{SUN}_\mathrm{smat}=0$.


In [ ]:
MODEL = "mattergen"
BINARY_DISTANCE = "smat"
CONTINUOUS_DISTANCE = "elmd+amd"


def load_metric_scores(path, model, distance, score_col):
	df = pd.read_csv(path)
	return df[(df["Model"] == model) & (df["Distance"] == distance)][
		score_col
	].to_numpy(dtype=float)


def require_same_length(**arrays):
	lengths = {name: len(values) for name, values in arrays.items()}
	if len(set(lengths.values())) != 1:
		raise ValueError(f"Mismatched input lengths: {lengths}")
	return next(iter(lengths.values()))


uni_path = JOURNAL_PREPROCESS_DIR / "uni.csv"
nov_path = JOURNAL_PREPROCESS_DIR / "nov.csv"

ehulls = np.asarray(load_gz(RESULTS_DIR / MODEL / "ehull.pkl.gz"), dtype=float)
binary_stability = (ehulls <= 0.1).astype(float)
continuous_stability = continuous_stability_scores(ehulls)

u_smat = load_metric_scores(uni_path, MODEL, BINARY_DISTANCE, "Uniqueness score")
n_smat = load_metric_scores(nov_path, MODEL, BINARY_DISTANCE, "Novelty score")
u_elmd_amd = load_metric_scores(
	uni_path, MODEL, CONTINUOUS_DISTANCE, "Uniqueness score"
)
n_elmd_amd = load_metric_scores(nov_path, MODEL, CONTINUOUS_DISTANCE, "Novelty score")

n_samples = require_same_length(
	ehulls=ehulls,
	u_smat=u_smat,
	n_smat=n_smat,
	u_elmd_amd=u_elmd_amd,
	n_elmd_amd=n_elmd_amd,
)

sun_smat = binary_stability * u_smat * n_smat
if not np.all(np.isin(sun_smat, [0.0, 1.0])):
	raise ValueError("SUN_smat is expected to be binary but contains other values")
sun_smat = sun_smat.astype(int)
csun_elmd_amd = continuous_stability * u_elmd_amd * n_elmd_amd

df_scores = pd.DataFrame(
	{
		"sample": np.arange(n_samples),
		"SUN_smat": sun_smat,
		"cSUN_elm+am": csun_elmd_amd,
	}
)
df_finite = df_scores[np.isfinite(df_scores["cSUN_elm+am"])].copy()
n_missing_csun = len(df_scores) - len(df_finite)

class_counts = df_finite["SUN_smat"].value_counts().sort_index()
if set(class_counts.index) != {0, 1}:
	raise ValueError(
		f"Probability of superiority requires both SUN_smat classes: {class_counts}"
	)

csun_0 = df_finite.loc[df_finite["SUN_smat"] == 0, "cSUN_elm+am"].to_numpy()
csun_1 = df_finite.loc[df_finite["SUN_smat"] == 1, "cSUN_elm+am"].to_numpy()
n_0 = len(csun_0)
n_1 = len(csun_1)
mann_whitney = mannwhitneyu(csun_1, csun_0, alternative="two-sided")
probability_of_superiority = mann_whitney.statistic / (n_1 * n_0)

summary = pd.DataFrame(
	[
		{
			"n total": len(df_scores),
			"n finite": len(df_finite),
			"n non-finite cSUN": n_missing_csun,
			"SUN_smat=0 count": int(n_0),
			"SUN_smat=1 count": int(n_1),
			"SUN_smat=0 cSUN mean": np.mean(csun_0),
			"SUN_smat=1 cSUN mean": np.mean(csun_1),
			"Probability of superiority": probability_of_superiority,
			"Mann-Whitney p-value": mann_whitney.pvalue,
		},
	]
)
summary

## 2. AMD k sensitivity

Compare `d_amd` uniqueness, novelty, and SUN scores for AMD embedding lengths
`k = 25, 50, 100, 200`.


In [ ]:
AMD_K_VALUES = [25, 50, 100, 200]
AMD_K_LABELS = {k: f"$k_\\mathrm{{max}}={k}$" for k in AMD_K_VALUES}
AMD_K_METRICS = ["uni", "nov", "sun"]
AMD_K_MEAN_SCORES_PATH = PREPROCESS_DIR / "amd_k_model_means.csv"
AMD_K_SPEARMAN_PATH = PREPROCESS_DIR / "amd_k_spearman.csv"


def amd_matrix_path(model, metric, k):
	if k == 100:
		return RESULTS_DIR / model / f"mtx_{metric}_amd.pkl.gz"
	return RESULTS_DIR / model / f"mtx_{metric}_amd_{k}.pkl.gz"


def metric_scores(mtx, metric, chunk_rows=CHUNK_ROWS):
	n_rows = mtx.shape[0]
	scores = np.empty(n_rows, dtype=float)
	valid_ref = np.isfinite(mtx[0]) if metric == "nov" else None
	for start in range(0, n_rows, chunk_rows):
		stop = min(start + chunk_rows, n_rows)
		block = mtx[start:stop]
		if metric == "uni":
			scores[start:stop] = np.nansum(block, axis=1) / (n_rows - 1)
		elif metric == "nov":
			scores[start:stop] = np.nanmin(block[:, valid_ref], axis=1)
		else:
			raise ValueError(f"Unsupported metric: {metric}")
	return scores


def compute_amd_k_scores(overwrite=OVERWRITE):
	if not overwrite and AMD_K_MEAN_SCORES_PATH.exists():
		return pd.read_csv(AMD_K_MEAN_SCORES_PATH)

	mean_records = []
	for model in MODELS_TEST:
		model_scores = {"uni": {}, "nov": {}}
		for metric in ["uni", "nov"]:
			for k in AMD_K_VALUES:
				mtx = load_gz(amd_matrix_path(model, metric, k))
				scores = metric_scores(mtx, metric)
				model_scores[metric][k] = scores
				mean_records.append(
					{
						"Metric": metric,
						"Distance": "amd",
						"k": k,
						"k label": AMD_K_LABELS[k],
						"Model": model,
						"Mean score": score_mean(scores),
					}
				)
				del mtx
				gc.collect()

		stability = continuous_stability_scores(
			load_gz(RESULTS_DIR / model / "ehull.pkl.gz")
		)
		for k in AMD_K_VALUES:
			scores = stability * model_scores["uni"][k] * model_scores["nov"][k]
			mean_records.append(
				{
					"Metric": "sun",
					"Distance": "amd",
					"k": k,
					"k label": AMD_K_LABELS[k],
					"Model": model,
					"Mean score": score_mean(scores),
				}
			)

	means = pd.DataFrame(mean_records)
	means["Rank"] = np.nan
	for (_, _), idx in means.groupby(["Metric", "k"]).groups.items():
		idx_models = means.loc[idx][means.loc[idx, "Model"] != "test"].index
		means.loc[idx_models, "Rank"] = means.loc[idx_models, "Mean score"].rank(
			ascending=False,
			method="min",
		)
	means.to_csv(AMD_K_MEAN_SCORES_PATH, index=False)
	return means


def compute_amd_k_spearman(df_means, overwrite=OVERWRITE):
	if not overwrite and AMD_K_SPEARMAN_PATH.exists():
		return pd.read_csv(AMD_K_SPEARMAN_PATH)
	records = []
	for metric in AMD_K_METRICS:
		df = df_means[(df_means["Metric"] == metric) & (df_means["Model"] != "test")]
		pivot = df.pivot(index="Model", columns="k", values="Mean score")
		corr = pivot[AMD_K_VALUES].corr(method="spearman")
		for k_i in AMD_K_VALUES:
			for k_j in AMD_K_VALUES:
				records.append(
					{
						"Metric": metric,
						"Distance": "amd",
						"k i": k_i,
						"k j": k_j,
						"k i label": AMD_K_LABELS[k_i],
						"k j label": AMD_K_LABELS[k_j],
						"Spearman rho": corr.loc[k_i, k_j],
					}
				)
	df_spearman = pd.DataFrame(records)
	df_spearman.to_csv(AMD_K_SPEARMAN_PATH, index=False)
	return df_spearman


def amd_k_spearman_matrix(df_spearman, metric):
	df = df_spearman[df_spearman["Metric"] == metric]
	matrix = df.pivot(index="k i", columns="k j", values="Spearman rho")
	matrix = matrix.loc[AMD_K_VALUES, AMD_K_VALUES]
	labels = [AMD_K_LABELS[k] for k in AMD_K_VALUES]
	matrix.index = labels
	matrix.columns = labels
	return matrix


df_amd_k_means = compute_amd_k_scores()
_ = plot_sensitivity_stripplots(
	df_amd_k_means,
	[(metric, "amd") for metric in AMD_K_METRICS],
	AMD_K_VALUES,
	lambda k: {"k": k},
	lambda k: AMD_K_LABELS[k],
	"amd_k_stripplots.pdf",
	figure_height=0.62,
	wspace=0.65,
	legend_y=-0.05,
	labelpad=PAD_L * 1.5,
)
plt.show()

In [ ]:
df_amd_k_spearman = compute_amd_k_spearman(df_amd_k_means)
_ = plot_spearman_heatmaps(
	[amd_k_spearman_matrix(df_amd_k_spearman, metric) for metric in AMD_K_METRICS],
	[metric_label(metric, "amd") for metric in AMD_K_METRICS],
	"amd_k_spearman_heatmaps.pdf",
	nrows=1,
	ncols=len(AMD_K_METRICS),
	figure_height=0.33,
	wspace=0.45,
)
plt.show()

## 3. Normalization sensitivity

Compare six normalizations for cached `elmd` and `amd` uniqueness and novelty
distance matrices.


In [ ]:
def normalize_cached_block(block, norm):
	y = np.clip(block, 0.0, 1.0 - np.finfo(float).eps)
	param = norm["param"]
	if norm["family"] == "frac":
		return np.divide(y, param + (1.0 - param) * y, out=y)
	denom = 1.0 - y
	np.divide(y, denom, out=y)
	y /= param
	np.negative(y, out=y)
	np.expm1(y, out=y)
	np.negative(y, out=y)
	return y


def normalized_metric_scores(mtx_cached, metric, norm, chunk_rows=CHUNK_ROWS):
	n_rows = mtx_cached.shape[0]
	scores = np.empty(n_rows, dtype=float)
	valid_ref = np.isfinite(mtx_cached[0]) if metric == "nov" else None
	for start in range(0, n_rows, chunk_rows):
		stop = min(start + chunk_rows, n_rows)
		block = mtx_cached[start:stop]
		if metric == "nov" and not np.all(valid_ref):
			block = block[:, valid_ref]
		block_norm = normalize_cached_block(block, norm)
		if metric == "uni":
			scores[start:stop] = np.nansum(block_norm, axis=1) / (n_rows - 1)
		else:
			scores[start:stop] = np.nanmin(block_norm, axis=1)
	return scores


def compute_normalized_scores(overwrite=OVERWRITE):
	if not overwrite and NORMALIZED_MEAN_SCORES_PATH.exists():
		return pd.read_csv(NORMALIZED_MEAN_SCORES_PATH)

	mean_records = []
	for metric in ["uni", "nov"]:
		for distance in DISTANCES:
			for model in MODELS_TEST:
				mtx = load_gz(RESULTS_DIR / model / f"mtx_{metric}_{distance}.pkl.gz")
				for norm in NORMALIZATIONS:
					scores = normalized_metric_scores(mtx, metric, norm)
					mean_records.append(
						{
							"Metric": metric,
							"Distance": distance,
							"Normalization": norm["key"],
							"Normalization label": norm["label"],
							"Model": model,
							"Mean score": score_mean(scores),
						}
					)
				del mtx
				gc.collect()

	means = pd.DataFrame(mean_records)
	means["Rank"] = np.nan
	for (_, _, _), idx in means.groupby(
		["Metric", "Distance", "Normalization"]
	).groups.items():
		idx_models = means.loc[idx][means.loc[idx, "Model"] != "test"].index
		means.loc[idx_models, "Rank"] = means.loc[idx_models, "Mean score"].rank(
			ascending=False,
			method="min",
		)
	means.to_csv(NORMALIZED_MEAN_SCORES_PATH, index=False)
	return means


def compute_normalization_spearman(df_means, overwrite=OVERWRITE):
	if not overwrite and NORMALIZATION_SPEARMAN_PATH.exists():
		return pd.read_csv(NORMALIZATION_SPEARMAN_PATH)
	records = []
	for metric in ["uni", "nov"]:
		for distance in DISTANCES:
			df = df_means[
				(df_means["Metric"] == metric)
				& (df_means["Distance"] == distance)
				& (df_means["Model"] != "test")
			]
			pivot = df.pivot(
				index="Model", columns="Normalization", values="Mean score"
			)
			corr = pivot[NORMALIZATION_KEYS].corr(method="spearman")
			for norm_i in NORMALIZATION_KEYS:
				for norm_j in NORMALIZATION_KEYS:
					records.append(
						{
							"Metric": metric,
							"Distance": distance,
							"Normalization i": norm_i,
							"Normalization j": norm_j,
							"Normalization i label": NORMALIZATION_LABELS[norm_i],
							"Normalization j label": NORMALIZATION_LABELS[norm_j],
							"Spearman rho": corr.loc[norm_i, norm_j],
						}
					)
	df_spearman = pd.DataFrame(records)
	df_spearman.to_csv(NORMALIZATION_SPEARMAN_PATH, index=False)
	return df_spearman


def normalization_spearman_matrix(df_spearman, metric, distance):
	df = df_spearman[
		(df_spearman["Metric"] == metric) & (df_spearman["Distance"] == distance)
	]
	matrix = df.pivot(
		index="Normalization i",
		columns="Normalization j",
		values="Spearman rho",
	)
	matrix = matrix.loc[NORMALIZATION_KEYS, NORMALIZATION_KEYS]
	labels = [NORMALIZATION_LABELS[key] for key in NORMALIZATION_KEYS]
	matrix.index = labels
	matrix.columns = labels
	return matrix


df_normalized_means = compute_normalized_scores()
_ = plot_sensitivity_stripplots(
	df_normalized_means,
	NORMALIZATION_PLOT_SETTINGS,
	NORMALIZATIONS,
	lambda norm: {"Normalization": norm["key"]},
	lambda norm: norm["label"],
	"normalize_stripplots.pdf",
	figure_height=0.8,
	wspace=0.8,
	legend_y=-0.02,
	labelpad=PAD_L,
)
plt.show()

In [ ]:
df_normalization_spearman = compute_normalization_spearman(df_normalized_means)
_ = plot_spearman_heatmaps(
	[
		normalization_spearman_matrix(df_normalization_spearman, metric, distance)
		for metric, distance in NORMALIZATION_PLOT_SETTINGS
	],
	[
		metric_label(metric, distance)
		for metric, distance in NORMALIZATION_PLOT_SETTINGS
	],
	"normalize_spearman_heatmaps.pdf",
	nrows=2,
	ncols=2,
	figure_height=0.9,
	wspace=0.45,
	hspace=0.55,
)
plt.show()

## 4. ElMD+AMD weight sensitivity

Vary the ElMD coefficient in `d_elmd+amd`, set the AMD coefficient to
`1 - w`, and compare model rankings for U, N, and SUN.


In [ ]:
def blended_metric_scores(
	mtx_elmd,
	mtx_amd,
	metric,
	coef_elmd,
	chunk_rows=CHUNK_ROWS,
):
	if mtx_elmd.shape != mtx_amd.shape:
		raise ValueError(f"Matrix shape mismatch: {mtx_elmd.shape} != {mtx_amd.shape}")
	coef_amd = 1.0 - coef_elmd
	n_rows = mtx_elmd.shape[0]
	scores = np.empty(n_rows, dtype=float)
	valid_ref = None
	if metric == "nov":
		valid_ref = np.isfinite(mtx_elmd[0]) & np.isfinite(mtx_amd[0])
	for start in range(0, n_rows, chunk_rows):
		stop = min(start + chunk_rows, n_rows)
		block = coef_elmd * mtx_elmd[start:stop] + coef_amd * mtx_amd[start:stop]
		if metric == "uni":
			scores[start:stop] = np.nansum(block, axis=1) / (n_rows - 1)
		elif metric == "nov":
			scores[start:stop] = np.nanmin(block[:, valid_ref], axis=1)
		else:
			raise ValueError(f"Unsupported metric: {metric}")
	return scores


def compute_elmd_amd_weight_scores(overwrite=OVERWRITE):
	if not overwrite and ELMD_AMD_WEIGHT_MEAN_SCORES_PATH.exists():
		return pd.read_csv(ELMD_AMD_WEIGHT_MEAN_SCORES_PATH)

	mean_records = []
	for model in MODELS_TEST:
		model_scores = {"uni": {}, "nov": {}}
		for metric in ["uni", "nov"]:
			mtx_elmd = load_gz(RESULTS_DIR / model / f"mtx_{metric}_elmd.pkl.gz")
			mtx_amd = load_gz(RESULTS_DIR / model / f"mtx_{metric}_amd.pkl.gz")
			for coef_elmd in ELMD_AMD_WEIGHTS:
				scores = blended_metric_scores(mtx_elmd, mtx_amd, metric, coef_elmd)
				model_scores[metric][coef_elmd] = scores
				mean_records.append(
					{
						"Metric": metric,
						"Distance": "elmd+amd",
						"ELMD_AMD_COEF_ELMD": coef_elmd,
						"ELMD_AMD_COEF_AMD": 1.0 - coef_elmd,
						"Model": model,
						"Mean score": score_mean(scores),
					}
				)
			del mtx_elmd, mtx_amd
			gc.collect()

		stability = continuous_stability_scores(
			load_gz(RESULTS_DIR / model / "ehull.pkl.gz")
		)
		for coef_elmd in ELMD_AMD_WEIGHTS:
			scores = (
				stability
				* model_scores["uni"][coef_elmd]
				* model_scores["nov"][coef_elmd]
			)
			mean_records.append(
				{
					"Metric": "sun",
					"Distance": "elmd+amd",
					"ELMD_AMD_COEF_ELMD": coef_elmd,
					"ELMD_AMD_COEF_AMD": 1.0 - coef_elmd,
					"Model": model,
					"Mean score": score_mean(scores),
				}
			)

	means = pd.DataFrame(mean_records)
	means["Rank"] = np.nan
	for (_, _), idx in means.groupby(["Metric", "ELMD_AMD_COEF_ELMD"]).groups.items():
		means.loc[idx, "Rank"] = means.loc[idx, "Mean score"].rank(
			ascending=False,
			method="min",
		)
	means.to_csv(ELMD_AMD_WEIGHT_MEAN_SCORES_PATH, index=False)
	return means


def plot_weight_rankings(
	df_means,
	filename="elmd_amd_weight_ranks.pdf",
):
	df = df_means.copy()
	df["Model label"] = df["Model"].map(MODEL_TEST_LABELS)
	df["Metric label"] = df["Metric"].map(
		lambda metric: metric_label(metric, "elmd+amd")
	)
	g = sns.relplot(
		data=df,
		x="ELMD_AMD_COEF_ELMD",
		y="Rank",
		hue="Model label",
		col="Metric label",
		kind="line",
		marker="o",
		markersize=5,
		linewidth=1,
		facet_kws={"sharey": False},
		hue_order=[MODEL_TEST_LABELS[model] for model in MODELS_TEST],
		col_order=[
			metric_label(metric, "elmd+amd") for metric in ELMD_AMD_WEIGHT_METRICS
		],
	)
	g.fig.set_size_inches(FIG_WIDTH, 0.25 * FIG_WIDTH)
	for ax in g.axes.flat:
		ax.axvline(ELMD_AMD_DEFAULT_WEIGHT, color="black", linewidth=1, linestyle="--")
		ax.invert_yaxis()
		ax.set_yticks(range(1, len(MODELS_TEST) + 1))
		ax.xaxis.set_major_locator(ticker.FixedLocator(ELMD_AMD_WEIGHTS))
		ax.tick_params(axis="both", labelsize=FONT_M)
		ax.set_xlabel(r"$w_\mathrm{elm}$", fontsize=FONT_L)
		ax.set_title(ax.get_title().split(" = ")[-1], fontsize=FONT_L, pad=PAD_M)
	g._legend.set_title("")
	legend_handles = [*g._legend.legend_handles]
	legend_labels = [text.get_text() for text in g._legend.get_texts()]
	legend_handles.append(
		plt.Line2D(
			[0],
			[0],
			color="black",
			linewidth=1,
			linestyle="--",
			label="Default weight",
		)
	)
	legend_labels.append("Default weight")
	g._legend.remove()
	g.fig.legend(
		handles=legend_handles,
		labels=legend_labels,
		loc="center right",
		frameon=False,
		fontsize=FONT_L,
	)
	g.axes.flat[0].set_ylabel("Rank", fontsize=FONT_L)
	g.fig.subplots_adjust(right=0.74, wspace=0.15)
	plt.savefig(FIGURES_DIR / filename, bbox_inches="tight")
	return g


df_elmd_amd_weight_means = compute_elmd_amd_weight_scores()
_ = plot_weight_rankings(df_elmd_amd_weight_means)
plt.show()

## 5. Stability shape sensitivity

Analyze model rankings for continuous stability scores using different
$\tau$ values and exponent shapes.


In [ ]:
STABILITY_TAUS = [0.1, 0.2, 0.3, 0.4, 0.5]
STABILITY_SHAPE_PLOT_TAU = 0.3
STABILITY_INTERCEPT_REFERENCE = 0.4289
STABILITY_EXPONENTS = [0.5, 1.0, 2.0]
STABILITY_EXPONENT_LABELS = {
	0.5: r"$\text{clip}(1 - \frac{E_\mathrm{hull}}{\tau}, 0, 1)^{0.5}$",
	1.0: r"$\text{clip}(1 - \frac{E_\mathrm{hull}}{\tau}, 0, 1)$",
	2.0: r"$\text{clip}(1 - \frac{E_\mathrm{hull}}{\tau}, 0, 1)^2$",
}
STABILITY_SHAPE_MEAN_SCORES_PATH = PREPROCESS_DIR / "stability_shape_model_means.csv"


def elmd_amd_scores_by_model(path, score_col):
	df = pd.read_csv(path)
	df = df[df["Distance"] == "elmd+amd"]
	return {
		model: df[df["Model"] == model][score_col].to_numpy(dtype=float)
		for model in MODELS_TEST
	}


def compute_stability_shape_scores(overwrite=OVERWRITE):
	if not overwrite and STABILITY_SHAPE_MEAN_SCORES_PATH.exists():
		return pd.read_csv(STABILITY_SHAPE_MEAN_SCORES_PATH)

	uni_scores = elmd_amd_scores_by_model(uni_path, "Uniqueness score")
	nov_scores = elmd_amd_scores_by_model(nov_path, "Novelty score")
	mean_records = []
	for model in MODELS_TEST:
		ehulls = np.asarray(load_gz(RESULTS_DIR / model / "ehull.pkl.gz"), dtype=float)
		if len(ehulls) != len(uni_scores[model]) or len(ehulls) != len(
			nov_scores[model]
		):
			raise ValueError(f"Length mismatch for {model}")
		for exponent in STABILITY_EXPONENTS:
			for tau in STABILITY_TAUS:
				stability = continuous_stability_scores(ehulls, tau, exponent)
				sun = stability * uni_scores[model] * nov_scores[model]
				mean_records.append(
					{
						"Metric": "sun",
						"Distance": "elmd+amd",
						"tau": tau,
						"Exponent": exponent,
						"Function family": STABILITY_EXPONENT_LABELS[exponent],
						"Model": model,
						"Mean score": score_mean(sun),
					}
				)

	means = pd.DataFrame(mean_records)
	means["Rank"] = np.nan
	for (_, _), idx in means.groupby(["Exponent", "tau"]).groups.items():
		means.loc[idx, "Rank"] = means.loc[idx, "Mean score"].rank(
			ascending=False,
			method="min",
		)
	means.to_csv(STABILITY_SHAPE_MEAN_SCORES_PATH, index=False)
	return means


def plot_stability_shape_functions(
	filename="stability_shape_functions_tau_0p3.pdf",
):
	x = np.linspace(-0.1, 0.5, 1000)
	fig, ax = plt.subplots(figsize=(0.7 * FIG_WIDTH, 0.4 * FIG_WIDTH))
	for exponent in STABILITY_EXPONENTS:
		y = continuous_stability_scores(x, STABILITY_SHAPE_PLOT_TAU, exponent)
		ax.plot(
			x,
			y,
			label=STABILITY_EXPONENT_LABELS[exponent],
			linestyle="-",
			linewidth=2,
		)
	ax.set_xlim(-0.1, 0.5)
	ax.set_ylim(-0.02, 1.02)
	ax.set_xlabel(r"$E_\mathrm{hull}$ [eV/atom]", fontsize=FONT_L, labelpad=PAD_S)
	ax.set_ylabel("Stability score", fontsize=FONT_L)
	ax.legend(fontsize=FONT_M)
	ax.tick_params(axis="both", labelsize=FONT_M)
	plt.savefig(FIGURES_DIR / filename, bbox_inches="tight")
	return fig


def plot_stability_shape_rankings(
	df_means,
	filename="stability_shape_ranks.pdf",
):
	df = df_means.copy()
	df["Model label"] = df["Model"].map(MODEL_TEST_LABELS)
	g = sns.relplot(
		data=df,
		x="tau",
		y="Rank",
		hue="Model label",
		col="Function family",
		kind="line",
		marker="o",
		markersize=5,
		linewidth=1,
		facet_kws={"sharey": False},
		hue_order=[MODEL_TEST_LABELS[model] for model in MODELS_TEST],
		col_order=[STABILITY_EXPONENT_LABELS[p] for p in STABILITY_EXPONENTS],
	)
	g.fig.set_size_inches(FIG_WIDTH, 0.25 * FIG_WIDTH)
	for ax in g.axes.flat:
		ax.invert_yaxis()
		ax.set_yticks(range(1, len(MODELS_TEST) + 1))
		ax.xaxis.set_major_locator(ticker.FixedLocator(STABILITY_TAUS))
		ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.1f"))
		ax.tick_params(axis="both", labelsize=FONT_M)
		ax.set_xlabel(r"$\tau$ [eV/atom]", fontsize=FONT_L)
		ax.set_title(ax.get_title().split(" = ")[-1], fontsize=FONT_L, pad=PAD_M)
	g.axes.flat[1].axvline(
		STABILITY_INTERCEPT_REFERENCE,
		color="black",
		linewidth=1,
		linestyle="--",
	)
	g._legend.set_title("")
	legend_handles = [*g._legend.legend_handles]
	legend_labels = [text.get_text() for text in g._legend.get_texts()]
	legend_handles.append(
		plt.Line2D(
			[0],
			[0],
			color="black",
			linewidth=1,
			linestyle="--",
			label=r"Default $\tau$",
		)
	)
	legend_labels.append(r"Default $\tau$")
	g._legend.remove()
	g.fig.legend(
		handles=legend_handles,
		labels=legend_labels,
		loc="center right",
		frameon=False,
		fontsize=FONT_L,
	)
	g.axes.flat[0].set_ylabel("Rank", fontsize=FONT_L)
	g.fig.subplots_adjust(right=0.74, wspace=0.15)
	plt.savefig(FIGURES_DIR / filename, bbox_inches="tight")
	return g


df_stability_shape_means = compute_stability_shape_scores()
_ = plot_stability_shape_functions()
plt.show()
_ = plot_stability_shape_rankings(df_stability_shape_means)
plt.show()

## 6. SUN component weight sensitivity

Analyze model rankings as product exponents $w_S$, $w_U$, and $w_N$ vary over
half-step triples with $w_S + w_U + w_N = 3$.


In [ ]:
SUN_WEIGHT_VALUES = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
SUN_WEIGHT_TOTAL = 3.0
SUN_WEIGHT_MEAN_SCORES_PATH = PREPROCESS_DIR / "sun_component_weight_model_means.csv"


def component_power(values, weight):
	values = np.asarray(values, dtype=float)
	if weight == 0:
		return np.ones_like(values, dtype=float)
	return np.nan_to_num(values, nan=0.0) ** weight


def valid_weight_triples():
	return [
		(w_s, w_u, w_n)
		for w_s in SUN_WEIGHT_VALUES
		for w_u in SUN_WEIGHT_VALUES
		for w_n in SUN_WEIGHT_VALUES
		if math.isclose(w_s + w_u + w_n, SUN_WEIGHT_TOTAL)
	]


def load_scores_by_model(path, score_col, distance="elmd+amd"):
	df = pd.read_csv(path)
	df = df[df["Distance"] == distance]
	return {
		model: df[df["Model"] == model][score_col].to_numpy(dtype=float)
		for model in MODELS_TEST
	}


def compute_sun_component_weight_scores(overwrite=OVERWRITE):
	if not overwrite and SUN_WEIGHT_MEAN_SCORES_PATH.exists():
		return pd.read_csv(SUN_WEIGHT_MEAN_SCORES_PATH)

	triples = valid_weight_triples()
	uni_scores = load_scores_by_model(uni_path, "Uniqueness score")
	nov_scores = load_scores_by_model(nov_path, "Novelty score")
	records = []
	for model in MODELS_TEST:
		stability = continuous_stability_scores(
			load_gz(RESULTS_DIR / model / "ehull.pkl.gz")
		)
		uni = uni_scores[model]
		nov = nov_scores[model]
		if not (len(stability) == len(uni) == len(nov)):
			raise ValueError(f"Length mismatch for {model}")
		for w_s, w_u, w_n in triples:
			scores = (
				component_power(stability, w_s)
				* component_power(uni, w_u)
				* component_power(nov, w_n)
			)
			records.append(
				{
					"Metric": "sun",
					"Distance": "elmd+amd",
					"w_S": w_s,
					"w_U": w_u,
					"w_N": w_n,
					"Model": model,
					"Model label": MODEL_TEST_LABELS[model],
					"Mean score": score_mean(scores),
				}
			)
	means = pd.DataFrame(records)
	means["Rank"] = np.nan
	for _, group in means.groupby(["w_S", "w_U", "w_N"]):
		idx = group.index
		means.loc[idx, "Rank"] = means.loc[idx, "Mean score"].rank(
			ascending=False,
			method="min",
		)
	means.to_csv(SUN_WEIGHT_MEAN_SCORES_PATH, index=False)
	return means


def ternary_xy(w_s, w_u, w_n):
	total = w_s + w_u + w_n
	x = (w_u + 0.5 * w_n) / total
	y = (math.sqrt(3) / 2 * w_n) / total
	return x, y


def draw_ternary_axes(ax):
	triangle = np.array([[0, 0], [1, 0], [0.5, math.sqrt(3) / 2], [0, 0]])
	ax.plot(triangle[:, 0], triangle[:, 1], color="black", linewidth=1)
	for value in SUN_WEIGHT_VALUES[1:-1]:
		fraction = value / SUN_WEIGHT_TOTAL
		ax.plot(
			[1 - fraction, 0.5 * (1 - fraction)],
			[0, math.sqrt(3) / 2 * (1 - fraction)],
			color="#D0D0D0",
			linewidth=0.5,
			zorder=0,
		)
		ax.plot(
			[fraction, 0.5 + 0.5 * fraction],
			[0, math.sqrt(3) / 2 * (1 - fraction)],
			color="#D0D0D0",
			linewidth=0.5,
			zorder=0,
		)
		ax.plot(
			[0.5 * fraction, 1 - 0.5 * fraction],
			[math.sqrt(3) / 2 * fraction, math.sqrt(3) / 2 * fraction],
			color="#D0D0D0",
			linewidth=0.5,
			zorder=0,
		)
	ax.text(-0.04, -0.04, r"$w_S=3$", ha="right", va="top", fontsize=FONT_M)
	ax.text(1.04, -0.04, r"$w_U=3$", ha="left", va="top", fontsize=FONT_M)
	ax.text(
		0.5,
		math.sqrt(3) / 2 + 0.04,
		r"$w_N=3$",
		ha="center",
		va="bottom",
		fontsize=FONT_M,
	)
	ax.set_xlim(-0.08, 1.08)
	ax.set_ylim(-0.08, math.sqrt(3) / 2 + 0.08)
	ax.set_aspect("equal")
	ax.axis("off")


def highlight_default_weight(ax):
	x, y = ternary_xy(1.0, 1.0, 1.0)
	ax.scatter(
		[x],
		[y],
		s=95,
		facecolors="none",
		edgecolors="black",
		linewidth=1.5,
		zorder=5,
	)


def plot_ternary_rank_models(
	means,
	filename="sun_component_weight_ternary.pdf",
):
	palette = {model: PALETTE[i] for i, model in enumerate(MODELS_TEST)}
	fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, 0.48 * FIG_WIDTH))
	for ax, (rank, title) in zip(
		axes,
		[(1, "Best model"), (2, "Second best model")],
		strict=False,
	):
		draw_ternary_axes(ax)
		ranked = means[means["Rank"] == rank].copy()
		x, y = ternary_xy(
			ranked["w_S"].to_numpy(),
			ranked["w_U"].to_numpy(),
			ranked["w_N"].to_numpy(),
		)
		ax.scatter(
			x,
			y,
			c=ranked["Model"].map(palette),
			s=28,
			edgecolor="black",
			linewidth=0.4,
			zorder=3,
		)
		highlight_default_weight(ax)
		ax.set_title(title, fontsize=FONT_L, pad=PAD_L)
	handles = [
		mpatches.Patch(color=palette[model], label=MODEL_TEST_LABELS[model])
		for model in MODELS_TEST
	]
	handles.append(
		plt.Line2D(
			[0],
			[0],
			marker="o",
			color="black",
			label=r"Default $(w_S=w_U=w_N=1)$",
			markerfacecolor="none",
			markersize=7,
			linewidth=0,
			markeredgewidth=1.5,
		)
	)
	fig.legend(handles=handles, loc="center right", frameon=False, fontsize=FONT_S)
	fig.subplots_adjust(right=0.76, wspace=0.40)
	fig.savefig(FIGURES_DIR / filename, bbox_inches="tight")
	return fig


df_sun_component_weight_means = compute_sun_component_weight_scores()
_ = plot_ternary_rank_models(df_sun_component_weight_means)
plt.show()

## 7. Sample-level cU and cN distributions

Plot sample-level continuous uniqueness and novelty score distributions for
`d_elmd+amd`.


In [ ]:
CU_LABEL = r"$\mathrm{cU}_\mathrm{elm{+}am}$"
CN_LABEL = r"$\mathrm{cN}_\mathrm{elm{+}am}$"
CU_CN_FIGURE_PATH = FIGURES_DIR / "sample_cu_cn_elmd_amd_distributions.pdf"


def load_sample_scores(path, score_col):
	df = pd.read_csv(path)
	df = df[df["Model"].isin(MODELS_TEST) & (df["Distance"] == "elmd+amd")].copy()
	df[score_col] = df[score_col].fillna(0.0)
	return df


def legend_handles_labels(ax):
	legend = ax.get_legend()
	if legend is None:
		return [], []
	handles = getattr(legend, "legend_handles", None)
	if handles is None:
		handles = legend.legendHandles
	labels = [text.get_text() for text in legend.get_texts()]
	return handles, labels


def plot_cu_cn_distributions(df_uni, df_nov):
	fig, axes = plt.subplots(
		1,
		2,
		figsize=(FIG_WIDTH, 0.38 * FIG_WIDTH),
		gridspec_kw={"wspace": 0.3},
	)
	plot_settings = [
		(axes[0], df_uni, "Uniqueness score", CU_LABEL, (0.75, 1.0)),
		(axes[1], df_nov, "Novelty score", CN_LABEL, (0.0, 1.0)),
	]
	legend_handles, legend_labels = [], []
	for i, (ax, df, score_col, score_label, xlim) in enumerate(plot_settings):
		sns.kdeplot(
			data=df,
			x=score_col,
			hue="Model",
			hue_order=MODELS_TEST,
			ax=ax,
			alpha=1.0,
			linewidth=0.5,
		)
		if i == 0:
			legend_handles, legend_labels = legend_handles_labels(ax)
		legend = ax.get_legend()
		if legend is not None:
			legend.remove()
		ax.set_xlim(*xlim)
		ax.set_xlabel(score_label, fontsize=FONT_L, labelpad=PAD_S)
		ax.set_ylabel("Density", fontsize=FONT_L)
		ax.tick_params(axis="both", labelsize=FONT_M)
	legend_labels = [MODEL_TEST_LABELS[label] for label in legend_labels]
	n = len(legend_handles) // 2
	legend_handles = [
		h for pair in zip(legend_handles[:n], legend_handles[n:]) for h in pair
	]
	legend_labels = [
		l for pair in zip(legend_labels[:n], legend_labels[n:]) for l in pair
	]
	fig.legend(
		legend_handles,
		legend_labels,
		loc="lower center",
		bbox_to_anchor=(0.5, -0.3),
		ncol=len(MODELS_TEST) // 2,
		fontsize=FONT_M,
		handletextpad=0.3,
	)
	fig.savefig(CU_CN_FIGURE_PATH, bbox_inches="tight")
	return fig


df_uni_distribution = load_sample_scores(uni_path, "Uniqueness score")
df_nov_distribution = load_sample_scores(nov_path, "Novelty score")
_ = plot_cu_cn_distributions(df_uni_distribution, df_nov_distribution)
plt.show()

## 8. Model-pair set distances

Compute set-level distances from sampled `d_elmd+amd` model-pair matrices and
plot energy distance and exact earth mover's distance.


In [ ]:
MODEL_PAIR_SAMPLE_SIZE = 1000
MODEL_PAIR_SEED = 0
MODEL_PAIR_CACHE_DIR = (
	PREPROCESS_DIR
	/ "model_pair_elmd_amd"
	/ f"n{MODEL_PAIR_SAMPLE_SIZE}_seed{MODEL_PAIR_SEED}"
)
MODEL_PAIR_FIGURE_PATH = FIGURES_DIR / "model_pair_set_distances.pdf"


def pair_matrix_path(model_1, model_2):
	return MODEL_PAIR_CACHE_DIR / f"mtx_{model_1}__{model_2}.pkl.gz"


def pair_matrix(model_1, model_2):
	if pair_matrix_path(model_1, model_2).exists():
		return np.asarray(load_gz(pair_matrix_path(model_1, model_2)), dtype=float)
	if pair_matrix_path(model_2, model_1).exists():
		return np.asarray(load_gz(pair_matrix_path(model_2, model_1)), dtype=float).T
	raise FileNotFoundError(
		f"Missing {model_1} vs {model_2} matrix in {MODEL_PAIR_CACHE_DIR}. "
		"Run compute_model_pair_elmd_amd.py first."
	)


def finite_mean(values):
	return np.nanmean(np.asarray(values, dtype=float)).item()


def energy_distance(model_1, model_2):
	d_xy = pair_matrix(model_1, model_2)
	d_xx = pair_matrix(model_1, model_1)
	d_yy = pair_matrix(model_2, model_2)
	return max(
		2 * finite_mean(d_xy) - finite_mean(d_xx) - finite_mean(d_yy),
		0.0,
	)


def earth_movers_distance(model_1, model_2):
	d_xy = pair_matrix(model_1, model_2)
	row_ind, col_ind = linear_sum_assignment(np.nan_to_num(d_xy, nan=np.inf))
	return finite_mean(d_xy[row_ind, col_ind])


def metric_table(metric_func):
	data = np.zeros((len(MODELS_TEST), len(MODELS_TEST)))
	for i, model_1 in enumerate(MODELS_TEST):
		for j, model_2 in enumerate(MODELS_TEST):
			data[i, j] = metric_func(model_1, model_2)
	labels = [MODEL_TEST_LABELS[model] for model in MODELS_TEST]
	return pd.DataFrame(data, index=labels, columns=labels)


def plot_set_distance_heatmaps(tables, filename=MODEL_PAIR_FIGURE_PATH):
	fig, axes = plt.subplots(
		1,
		2,
		figsize=(FIG_WIDTH, FIG_WIDTH * 0.5),
		constrained_layout=True,
	)
	for ax, (title, table) in zip(axes, tables.items(), strict=True):
		hm = sns.heatmap(
			table,
			ax=ax,
			cmap="viridis",
			square=True,
			annot=True,
			fmt=".3f",
			annot_kws={"fontsize": FONT_S - 1},
			cbar=True,
			cbar_kws={"shrink": 0.55, "pad": 0.05},
			linewidths=0.25,
			linecolor="white",
		)
		cbar = hm.collections[0].colorbar
		cbar.ax.tick_params(labelsize=FONT_M)
		ax.set_title(title, fontsize=FONT_L, pad=PAD_M)
		ax.set_xlabel("")
		ax.set_ylabel("")
		ax.tick_params(axis="x", labelrotation=90, labelsize=FONT_S)
		ax.tick_params(axis="y", labelrotation=0, labelsize=FONT_S)
	filename.parent.mkdir(parents=True, exist_ok=True)
	fig.savefig(filename, bbox_inches="tight")
	return fig


set_distance_tables = {
	"Energy distance": metric_table(energy_distance),
	"Earth mover's distance": metric_table(earth_movers_distance),
}
_ = plot_set_distance_heatmaps(set_distance_tables)
plt.show()